# Solutions · Chapter 04-06 · Preprocessing

Worked answers for `notebooks/04_workflow/04-06_preprocessing.ipynb`.

E9 and E10 both go against the chapter's implied moral, and are the two most useful answers here.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")


# SYNTHETIC. The rental listings from 04-06.
def load_flats():
    rng = np.random.default_rng(19)
    n = 1200
    levels = ["D%02d" % i for i in range(12)]
    district = rng.choice(levels, n, p=np.array([9, 8, 7, 6, 6, 5, 5, 4, 4, 3, 2, 1]) / 60)
    premium = dict(zip(rng.permutation(levels), np.linspace(1.35, 0.75, 12)))
    area = np.round(np.exp(rng.normal(4.0, 0.35, n)), 1)
    rooms = np.clip(np.round(area / 28 + rng.normal(0, 0.5, n)), 1, 6).astype(int)
    age = np.clip(np.round(rng.gamma(3.0, 14.0, n)), 0, 140).astype(int)
    has_lift = (rng.random(n) < 1 / (1 + np.exp(3 - 0.05 * (120 - age)))).astype(int)
    rent = np.round(np.maximum(150, 6.0 * area * np.array([premium[d] for d in district])
                               - 1.2 * age + 60 * has_lift + rng.normal(0, 60, n)), 0)
    flats = pd.DataFrame({"district": district, "area_m2": area, "rooms": rooms,
                          "age_years": age, "has_lift": has_lift, "rent_eur": rent})
    flats.loc[rng.random(n) < 0.05 + 0.45 * (flats.age_years > 60), "age_years"] = np.nan
    flats.loc[rng.random(n) < 0.12, "area_m2"] = np.nan
    return flats


flats = load_flats()
NUMERIC = ["area_m2", "rooms", "age_years", "has_lift"]
CATEGORICAL = ["district"]
target = flats.rent_eur
folds = KFold(5, shuffle=True, random_state=0)
numeric = make_pipeline(SimpleImputer(), StandardScaler())


def error_of(model, frame=None, labels=None):
    frame = flats[NUMERIC + CATEGORICAL] if frame is None else frame
    labels = target if labels is None else labels
    return -cross_val_score(model, frame, labels, cv=folds,
                            scoring="neg_mean_absolute_error").mean()


print("%d listings; baseline pipeline MAE %.2f"
      % (len(flats),
         error_of(make_pipeline(
             ColumnTransformer([("numeric", numeric, NUMERIC),
                                ("categorical", OneHotEncoder(handle_unknown="ignore"),
                                 CATEGORICAL)]), Ridge()))))

## Quick understanding

### E1

| Pattern | Means | Implication |
|---|---|---|
| **MCAR** - completely at random | the hole is unrelated to anything | impute with anything sensible; the choice barely matters and nothing is lost |
| **MAR** - at random *given other columns* | the hole depends on columns you have | impute **using those columns** - a model-based imputer can genuinely recover information |
| **MNAR** - not at random | the hole depends on the value that is missing | no imputation can recover it. Impute, **add an indicator**, and say so in the write-up |

### E2

Because the two numbers are computed on different rows, so they answer different questions. Dropping
incomplete rows removes exactly the rows that were hard - here the old buildings whose age records were
lost, which rent for 336 EUR against 365 for the survivors. **The metric improves because the problem got
easier, not because the method got better.**

On the same 915 complete rows both procedures score 51.41, because there is nothing left to impute.

### E3

**Can leak:** target encoding (it reads `y` directly), and any feature selection or supervised transform.

**Cannot leak:** imputation with a constant, mean or median; one-hot and ordinal encoding; scaling; log
and power transforms. All are functions of `X` alone.

**Still fit them on training data only** - the reason is the cheapness of the discipline, not the size of
the risk. 04-05 measured the risk for a scaler at +0.0005.

## Hand calculation

### E4

Values 10, 20, 30, missing, missing. The mean of the observed values is **20**, so after imputing:
10, 20, 30, 20, 20.

- **Before:** mean 20, standard deviation **10.0000** (over the three observed values, `ddof=1`).
- **After:** mean **20** - unchanged, which is the point of mean imputation - and standard deviation
  **7.0711**.

**The spread shrank by 29%.** Two invented values sitting exactly at the centre make the column look
tighter than it is.

**Why that matters for a distance method:** kNN compares columns by their spread. A column whose variance
has been artificially shrunk contributes *less* to every distance than it should, so mean-imputing a
column quietly reduces its influence on the neighbours chosen - by an amount that depends on how much of
it was missing. The more holes a column has, the less the model listens to it, which is precisely
backwards if the missingness was informative.

### E5

The three levels are encoded 1, 2, 3, and their true effects are +100, -50, +30.

A linear model with one coefficient `b` predicts `b*1`, `b*2`, `b*3` - three values in **arithmetic
progression**, evenly spaced and monotone. The true effects are neither: they go up, down, then up. No
single `b` can produce that pattern, and the best fit is a compromise close to zero.

**One-hot needs two coefficients** (three levels minus a reference, which the intercept absorbs), or
three without an intercept. With two free coefficients plus the intercept, all three effects are
representable exactly.

### E6

`exp(5.6), exp(6.0), exp(6.4)` = 270.4, 403.4, 601.8, with a **mean of 425.2**.

The true log-rents 5.5, 6.1, 6.4 give 244.7, 445.9, 601.8, with a **mean of 430.8**.

**The truth is larger**, by 5.6. And that matches the chapter: the predictions are less spread out in log
space than the truth is, and because `exp` is convex, spread *raises* the mean after exponentiating. A
model that shrinks predictions towards the middle - which every regularised model does - therefore
under-predicts the mean once you transform back.

### E7

`50000 / 8000` = **6.25 rows per level**.

By 04-05's rule, **target encoding is not safe here**: with about six rows per postcode, each row
contributes roughly a sixth of its own encoded value. Use it only with the encoding computed inside the
fold, heavy smoothing, and a check that it beats a simpler alternative.

**One-hot would add 8,000 columns** to 50,000 rows - a matrix wider than a fifth of its length, which is
the wide-data regime and brings 04-05's selection hazards with it.

The usual practical answers: group rare levels into "other", encode a coarser geography (the first few
characters of the postcode), or use a model that handles high-cardinality categories natively.

In [ ]:
observed = np.array([10.0, 20.0, 30.0])
imputed = np.array([10.0, 20.0, 30.0, 20.0, 20.0])
print("E4  before: mean %.1f, sd %.4f" % (observed.mean(), observed.std(ddof=1)))
print("    after : mean %.1f, sd %.4f  (spread down %.0f%%)"
      % (imputed.mean(), imputed.std(ddof=1),
         100 * (1 - imputed.std(ddof=1) / observed.std(ddof=1))))
print()
predicted = np.exp([5.6, 6.0, 6.4])
actual = np.exp([5.5, 6.1, 6.4])
print("E6  exp(predictions) %s -> mean %.1f" % (np.round(predicted, 1), predicted.mean()))
print("    exp(truth)       %s -> mean %.1f" % (np.round(actual, 1), actual.mean()))
print("    the truth is larger by %.1f" % (actual.mean() - predicted.mean()))
print()
print("E7  %d rows / %d levels = %.2f rows per level" % (50000, 8000, 50000 / 8000))

## Coding

### E8 - a missingness report

In [ ]:
def missingness_report(frame, labels):
    rows = []
    for column in frame.columns:
        absent = frame[column].isna()
        if not absent.any():
            continue
        present, missing = labels[~absent], labels[absent]
        gap = missing.mean() - present.mean()
        # 03-03: the standard error of a difference between two means
        standard_error = np.sqrt(present.var(ddof=1) / len(present) + missing.var(ddof=1) / len(missing))
        rows.append({"column": column,
                     "missing": int(absent.sum()),
                     "share": "%.1f%%" % (100 * absent.mean()),
                     "mean when present": round(present.mean(), 1),
                     "mean when missing": round(missing.mean(), 1),
                     "gap": round(gap, 1),
                     "standard error": round(standard_error, 1),
                     "gap / se": round(gap / standard_error, 2)})
    if not rows:
        print("no missing values")
        return None
    report = pd.DataFrame(rows)
    report["informative?"] = np.where(report["gap / se"].abs() > 2, "LIKELY", "probably not")
    return report.reindex(report["gap / se"].abs().sort_values(ascending=False).index)


print(missingness_report(flats, target).to_string(index=False))

**`age_years` is flagged and `area_m2` is not**, which is the conclusion the chapter's histograms
reached by eye.

The threshold is not a convention here - it is 03-03's standard error of a difference between two means,
`sqrt(var1/n1 + var2/n2)`. A gap of 32.8 EUR on a standard error of 13.8 is **2.38 standard errors**, so
it is unlikely to be sampling noise. `area_m2`'s gap of 13.5 on a standard error of 12.2 is **1.11** -
entirely consistent with chance.

That matters because the raw gaps look similar in size, and a threshold based on the target's spread
would have missed the difference. **Two columns can have the same missing rate and gaps of the same order,
and only one of them can be telling you something** - the second half of E20's point, arriving early.

### E9 - do cleverer imputers help?

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

import time

rows = []
for label, imputer in [("mean", SimpleImputer()),
                       ("mean + indicator", SimpleImputer(add_indicator=True)),
                       ("KNNImputer, k=5", KNNImputer(n_neighbors=5)),
                       ("IterativeImputer", IterativeImputer(random_state=0, max_iter=10))]:
    model = make_pipeline(
        ColumnTransformer([("numeric", make_pipeline(imputer, StandardScaler()), NUMERIC),
                           ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
        Ridge())
    started = time.perf_counter()
    score = error_of(model)
    rows.append({"imputer": label, "MAE": round(score, 2),
                 "seconds for 5 folds": round(time.perf_counter() - started, 2)})
imputer_table = pd.DataFrame(rows)
imputer_table["better than the mean by"] = (imputer_table.MAE.iloc[0] - imputer_table.MAE).round(2)
print(imputer_table.to_string(index=False))

**Yes, and by more than anything else in the chapter.** `KNNImputer` scores **55.11** against the mean's
58.50 - a gain of **3.39 EUR**, five times what the missing indicator was worth.

That is worth taking seriously, because the chapter's summary figure understates it. The reason the
cleverer imputers win here is visible in how the data was built: **`area_m2` is missing completely at
random, and it is strongly related to `rooms` and to `rent`.** A neighbour-based imputer can look at a
flat's rooms and district and reconstruct a plausible area; the column mean cannot. That is the **MAR**
case from E1, and it is exactly where model-based imputation is supposed to pay.

**What it costs:** the `seconds` column. `KNNImputer` is markedly slower than a mean, and it must be
refitted inside every fold and stored with the model, since at prediction time it needs the training rows
to find neighbours in.

So the honest summary is a trade, not a rule:

- If missingness is **MCAR or MAR** and the column is well predicted by others, a model-based imputer can
  earn several times what an indicator earns. **Measure it; do not assume the mean is fine.**
- If missingness is **MNAR**, no imputer recovers the value, and the indicator is the part that matters.
- The mean remains the right default for a **first** pipeline, because it is instant and it establishes
  the baseline the cleverer version must beat - which is 04-02's argument applied to preprocessing.

### E10 - an ordinal encoding with a meaningful order

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class RankByTarget(BaseEstimator, TransformerMixin):
    # orders the levels by their mean target ON THE TRAINING FOLD, then encodes the rank.
    def fit(self, X, y):
        means = pd.Series(np.asarray(y)).groupby(np.asarray(X).ravel()).mean().sort_values()
        self.rank_ = {level: position for position, level in enumerate(means.index)}
        self.default_ = len(means) / 2
        return self

    def transform(self, X):
        return np.array([[self.rank_.get(value, self.default_)] for value in np.asarray(X).ravel()],
                        dtype=float)


ranked = make_pipeline(
    ColumnTransformer([("numeric", numeric, NUMERIC), ("categorical", RankByTarget(), "district")]),
    Ridge())

print("ordinal, ranked by training-fold mean rent : MAE %.2f" % error_of(ranked))
print("one-hot, 12 columns                        : MAE 58.50")
print("ordinal, alphabetical                      : MAE 75.92")
print("district dropped entirely                  : MAE 77.75")
print()
print("share of the district's value recovered by one ranked column: %.0f%%"
      % (100 * (77.75 - error_of(ranked)) / (77.75 - 58.50)))

**59.65 - one column recovering 94% of what twelve columns achieve.**

Ordinal encoding was never the problem. **An arbitrary order was.** Given an order that means something,
a single column carries nearly all of the district's information, and the 9.5% figure from the chapter is
a fact about alphabetical district codes rather than about ordinal encoding.

**What you have just re-invented is target encoding** - in its ranked, ordinal form. Note that
`RankByTarget.fit` takes `y`, which makes it exactly the kind of step 04-05 said must live inside the
fold, and putting it in a `Pipeline` is what makes that automatic. Fitted on all the data it would leak;
fitted per fold, as here, it is legitimate and cheap.

The remaining 6% is what one-hot buys: **ranks are still evenly spaced**, so a district that is far more
expensive than its neighbour in the ordering cannot be represented as far away. One-hot has no such
constraint, at the price of eleven extra columns.

### E11 - the whole pipeline, and the deliberate break

In [ ]:
# like for like: the SAME imputer and scaler in both, only their fitting scope differs
correct = make_pipeline(
    ColumnTransformer([
        ("numeric", make_pipeline(SimpleImputer(), StandardScaler()), NUMERIC),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
    Ridge())
print("everything inside the pipeline        : MAE %.2f" % error_of(correct))

# deliberately break it: fit the scaler on all the data, before any splitting
leaked = flats.copy()
leaked[NUMERIC] = StandardScaler().fit_transform(SimpleImputer().fit_transform(flats[NUMERIC]))
broken = make_pipeline(
    ColumnTransformer([("numeric", "passthrough", NUMERIC),
                       ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
    Ridge())
print("scaler and imputer fitted on all data : MAE %.2f"
      % error_of(broken, leaked[NUMERIC + CATEGORICAL], target))

**The difference is a few hundredths of a EUR** - 04-05's +0.0005 result, reproduced on a different
dataset with a different model.

**And you should still not do it**, for three reasons that have nothing to do with this number:

1. **It does not generalise.** The step that is safe today is a scaler. Tomorrow somebody adds a target
   encoder or a feature selector to the same block, and it will not be safe, and nothing will warn them.
2. **The pipeline is what makes the correct thing the default.** A `Pipeline` refits every step per fold
   automatically. Doing it by hand means being right every time, on every project, under deadline.
3. **A fitted-outside step cannot be deployed.** The pipeline object carries its own preprocessing; a
   loose scaler in a notebook has to be remembered, serialised and reapplied in the same order at
   prediction time, which is a real and common source of production bugs.

The measurement tells you how urgently to fix a legacy codebase. It does not tell you how to write a new
one.

### E12 - counting unseen categories

In [ ]:
class CountingOneHot(BaseEstimator, TransformerMixin):
    def __init__(self, warn_above=0.01):
        self.warn_above = warn_above

    def fit(self, X, y=None):
        self.encoder_ = OneHotEncoder(handle_unknown="ignore").fit(X)
        self.known_ = set(self.encoder_.categories_[0])
        return self

    def transform(self, X):
        values = np.asarray(X).ravel()
        unseen = np.array([value not in self.known_ for value in values])
        self.last_unseen_share_ = float(unseen.mean())
        if self.last_unseen_share_ > self.warn_above:
            print("  WARNING: %.1f%% of rows (%d of %d) have an unseen category"
                  % (100 * self.last_unseen_share_, unseen.sum(), len(values)))
        return self.encoder_.transform(X)


held_back = flats[flats.district != "D11"]
arriving = flats[flats.district == "D11"]

counter = CountingOneHot().fit(held_back[CATEGORICAL])
print("predicting on rows the model has seen the districts of:")
counter.transform(held_back[CATEGORICAL].head(50))
print("  (silent - %.1f%% unseen)" % (100 * counter.last_unseen_share_))
print("predicting on the %d D11 listings:" % len(arriving))
counter.transform(arriving[CATEGORICAL])

The point of the warning is that **`handle_unknown="ignore"` is silent by design**, and silence is the
wrong default for a value the model has never seen. In production this counter becomes a monitored
metric, and a rising share of unknown categories is one of the earliest signals that the world has moved
away from the training data - which is 13-04's subject.

## Interpretation

### E13

**"On which rows?"**

An improvement from 58 to 41 is enormous - 29% - and preprocessing does not usually produce that. The
most likely explanation is 04-06's trap: a model-based imputer that fails on some rows, or a step that
drops rows it cannot handle, changes the row set, and the new number is measured on an easier subset.

The second question, if the rows are the same: **"was the imputer fitted inside the folds?"** A
`IterativeImputer` fitted on all the data has seen the test rows' feature values - which is preprocessing
leakage of the mild, `X`-only kind and usually small, but an imputer that uses the *target* as an input
column is a different matter entirely and would explain a 29% jump exactly.

Only after both would I look at whether the gain is real. It might be - E9 found 3.39 EUR from
`KNNImputer` on this data - but 17 EUR needs an explanation before it needs congratulations.

### E14

**What it has learned:** with one row per customer, the target encoding of `customer_id` is that
customer's own target value. The column is a perfect copy of `y`, and the model will fit it with weight
close to 1 and ignore everything else.

**What it will do in production:** every customer is new, so every `customer_id` is unseen, so every row
gets the fallback value - the global mean. The model will predict the same number for everybody, and the
offline metric will have been near-perfect. This is 04-05's target-encoding leak at its theoretical
maximum, and it is not rare: an id column looks like a categorical column to any automated pipeline.

## Debugging

### E15

**The likely cause: a high-cardinality column was one-hot encoded**, adding thousands of columns. Training
time scales with the width of the matrix, and the score got *worse* because thousands of sparse
near-empty columns are thousands of opportunities to fit noise.

**Two fixes:**

1. **Group rare levels.** Keep the levels covering, say, 95% of rows and map the rest to "other". Usually
   recovers almost all of the signal at a fraction of the width.
2. **Use a target encoding computed inside the fold**, which is one column - E10 showed a ranked version
   recovering 94% of one-hot's value here with a twelfth of the columns.

A third, if the model allows it: use a gradient-boosting implementation with native categorical support,
which splits on categories without expanding them.

### E16

1. **Unseen categories at predict time.** One-hot encoding fitted on training data produces a fixed number
   of columns; new data with a new level either errors or, if you one-hot encoded outside a pipeline with
   `pd.get_dummies`, produces a *different number of columns* - and the model receives the wrong shape.
   This is the single commonest cause, and it is why `pd.get_dummies` should not be used as a pipeline
   step.
2. **Column order or column set differs.** The production frame has the columns in a different order, or
   is missing one, or has an extra one. A `ColumnTransformer` addressing columns **by name** is robust to
   order; positional indexing is not.

Both disappear if the preprocessing lives inside the fitted pipeline object, which is the argument E11
was making.

## Exam and interview reasoning

### E17

> "First I look at *where* the values are missing and whether the missingness is informative - I compare
> the target between rows where the value is present and rows where it is not. If there is a gap, the
> hole is carrying information, so I impute and add a binary indicator column. If there is no gap, I
> impute and move on. For the imputation itself I start with the median as a baseline and check whether a
> model-based imputer beats it, because sometimes it does substantially. And all of it goes inside the
> pipeline so it is refitted on each training fold."

**"Why not just drop those rows?"**

> "Two reasons. It usually removes a biased subset - on the data I was working with, the rows with
> missing values were systematically cheaper, so dropping them made the evaluation look better while
> making the model worse on exactly the cases it would meet. And the second reason is that those rows
> still arrive in production. You cannot decline to predict for a customer because a field is blank, so
> the model has to handle missingness one way or another; dropping just means deciding not to decide."

## Transfer to a different situation

### E18

| Column | Decision | Reasoning |
|---|---|---|
| `blood_pressure`, 30% missing | impute (median or model-based) **plus an indicator** | 30% is a lot, and in clinical data "not measured" usually means "not thought necessary" or "the patient was too unwell" - both informative. Test it |
| `ward_code`, 60 levels | one-hot if the model tolerates 60 columns; otherwise group rare wards or target-encode inside the fold | 60 is borderline. Check rows per level first |
| `days_since_last_admission`, missing when there was no previous admission | **do not impute a number.** Add an indicator `first_admission` and fill with a sentinel the model can isolate | this is not really missing data |
| `cost_eur`, heavily skewed | consider a log transform **if it is a feature**; if it is the target, remember the back-transform bias | test rather than assume |

**`days_since_last_admission` is the interesting one, and it is arguably none of the three patterns.** The
value is not missing - **it does not exist**. There is no true number being hidden, so MCAR, MAR and MNAR
are all the wrong frame. Imputing a median here invents a previous admission that never happened.

The correct treatment is to recognise that the column is really two pieces of information - *has this
patient been admitted before?* and *if so, how long ago?* - and to encode both. This case is common
enough to be worth naming: **"structurally missing" or "not applicable"**, and mistaking it for ordinary
missingness is one of the most frequent preprocessing errors in real datasets.

## Explain it to someone non-technical

### E19

> "Suppose a survey asks for people's income and a fifth leave it blank. If we write the average income
> into every blank, we have not recovered anything - we have invented a fifth of our data, and made it
> unusually consistent. Worse, people often leave income blank for a reason, so the blanks are not
> random: we have replaced a meaningful silence with an average person who does not exist. It is
> sometimes the right thing to do, but it is a decision about the data, not a cleaning step."

(88 words.)

## Optional challenge

### E20 - proving that the pattern, not the rate, is what matters

In [ ]:
def make_pair(rows=1500, seed=0):
    # identical data; the ONLY difference is which rows lose their x1 value
    rng = np.random.default_rng(seed)
    x1 = rng.normal(size=rows)
    x2 = rng.normal(size=rows)
    y = 3 * x1 + 2 * x2 + rng.normal(0, 1, rows)

    mcar = pd.DataFrame({"x1": x1, "x2": x2})
    mcar.loc[rng.random(rows) < 0.30, "x1"] = np.nan          # missing for no reason

    mnar = pd.DataFrame({"x1": x1, "x2": x2})
    mnar.loc[x1 > np.quantile(x1, 0.70), "x1"] = np.nan       # missing when x1 is large
    return mcar, mnar, pd.Series(y)


def score(frame, labels, with_indicator):
    model = make_pipeline(SimpleImputer(add_indicator=with_indicator), StandardScaler(), Ridge())
    return -cross_val_score(model, frame, labels, cv=folds,
                            scoring="neg_mean_absolute_error").mean()


mcar_frame, mnar_frame, labels = make_pair()
print("both datasets have exactly 30%% of x1 missing: %.1f%% and %.1f%%"
      % (100 * mcar_frame.x1.isna().mean(), 100 * mnar_frame.x1.isna().mean()))
print()
rows = []
for pattern, frame in [("MCAR (missing for no reason)", mcar_frame),
                       ("MNAR (missing when x1 is large)", mnar_frame)]:
    without = score(frame, labels, False)
    with_it = score(frame, labels, True)
    rows.append({"pattern": pattern, "no indicator": round(without, 4),
                 "with indicator": round(with_it, 4), "the indicator is worth": round(without - with_it, 4)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.3), sharey=True)
for ax, (title, frame) in zip((left, right),
                              [("MCAR: missing for no reason", mcar_frame),
                               ("MNAR: missing when x1 is large", mnar_frame)]):
    absent = frame.x1.isna()
    ax.hist(labels[~absent], bins=34, alpha=0.75, color="#0072B2", density=True, label="x1 present")
    ax.hist(labels[absent], bins=34, alpha=0.75, color="#D55E00", density=True, label="x1 missing")
    ax.set_xlabel("target")
    ax.set_title("%s\ngap between the means: %.2f"
                 % (title, labels[absent].mean() - labels[~absent].mean()), fontsize=10.5)
    ax.legend(fontsize=8)
left.set_ylabel("density")
fig.suptitle("Same 30% missing rate. Only one of them tells you anything", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

**Exactly one of the four combinations rewards the indicator**, and it is MNAR - as designed.

Both datasets have the same 30% missing rate, the same columns and the same relationship. **The only
difference is *which* rows lost their value**, and it changes whether a binary column is worth adding
from nothing to a substantial gain.

The two histograms are the diagnostic from the chapter, run on data where the answer is known. On the
left the two distributions sit on top of each other - the hole says nothing. On the right they are
visibly separated, because rows are missing precisely when `x1` was large, and large `x1` means a large
target.

**That is the whole lesson, and it is why the rate is the wrong thing to look at.** "This column is 30%
missing" tells you how much you lost. It does not tell you whether what remains can be recovered, and the
histogram costs three lines.